# Tabac et cancer du poumon : d'où viennent les écarts entre départements ?

*Projet Python pour la data science — ENSAE 2A — Jean Palazuelo, Ilyan Radjabaly*

## Introduction

Le tabac est la première cause de cancer du poumon, c'est établi depuis longtemps. Ce qui nous intéresse ici, c'est autre chose : la fréquence de ce cancer varie beaucoup d'un département à l'autre, et on voulait savoir si ces écarts se retrouvent dans la géographie du tabagisme.

Deux difficultés guident tout le projet :
- un cancer du poumon apparaît en général 20 à 30 ans après l'exposition, donc il faut comparer le cancer d'aujourd'hui au tabac d'avant ;
- les départements n'ont pas la même pyramide des âges, et ce cancer touche surtout les plus de 60 ans. Sans correction, on comparerait surtout l'âge des habitants.

**Question :** à âge égal, les départements les plus touchés par le cancer du poumon sont-ils ceux des régions où l'on fumait le plus ?

## 1. Les données

Trois sources, toutes téléchargées par le code (fonctions dans `src/data/`) et mises en cache dans `data/raw/` :

| Données | Source | Niveau | Années |
|---|---|---|---|
| Patients traités pour un cancer du poumon, par classe d'âge de 5 ans | Assurance Maladie, data.ameli.fr (API) | département et région | 2015-2024 |
| Part de fumeurs quotidiens chez les adultes, avec intervalle de confiance | Santé publique France, Baromètre santé (API Odissé) | région | 2000, 2005, 2010, 2014, 2017, 2021 |
| Contours des départements | france-geojson (IGN) | département | — |

À noter dès maintenant :
- la donnée Ameli compte les personnes **prises en charge** pour un cancer du poumon une année donnée (une prévalence), pas les nouveaux cas ni les décès ;
- le tabagisme n'existe qu'au **niveau régional**, et pas pour la Corse (échantillon trop petit).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data.ameli import telecharger_cancer_poumon
from src.data.tabac import telecharger_tabac
from src.data.geo import telecharger_departements
from src.cleaning import taux_cancer, tabac_par_region, construire_table
from src.viz import cartes_cote_a_cote, nuage, evolution, nuage_ic, barres

ameli = telecharger_cancer_poumon()
tabac = telecharger_tabac()
contours = telecharger_departements()

print(ameli.shape, tabac.shape, contours.shape)
ameli.head()

La table Ameli a une ligne par année, département et classe d'âge : `ntop` est le nombre de patients, `npop` la population couverte. Les lignes `dept = "999"` donnent le total de chaque région, et `cla_age_5 = "tsage"` le total tous âges.

`ntop` est parfois vide : quand l'effectif est trop petit, l'Assurance Maladie ne le publie pas (secret statistique). Regardons où ça arrive :

In [ ]:
metro = ameli[~ameli["dept"].str.startswith("97") & (ameli["dept"] != "999") & (ameli["cla_age_5"] != "tsage")]
metro.groupby("cla_age_5")["ntop"].apply(lambda s: s.isna().mean()).round(2)

Les cases masquées sont presque toutes chez les moins de 40 ans, où il n'y a quasiment pas de cancers du poumon : les compter comme 0 ne change presque rien. C'est moins vrai pour les très âgés (près de la moitié des cases masquées chez les 90-94 ans, presque toutes chez les 95 ans et plus) : on sous-estime donc un peu le taux des petits départements, mais ces classes pèsent très peu une fois pondérées par la population.

## 2. Nettoyage : un taux comparable entre départements

On ne peut pas comparer des nombres de patients (le Nord a 2,5 millions d'habitants, la Lozère 76 000). On calcule donc un taux pour 100 000 habitants, et surtout un **taux standardisé sur l'âge** :

1. dans chaque département, on calcule le taux de chaque classe d'âge ;
2. on fait la moyenne de ces taux en les pondérant par la part de chaque classe d'âge dans la France métropolitaine.

On obtient le taux qu'aurait le département s'il avait la même pyramide des âges que la métropole. On fait la moyenne sur 2022-2024 pour lisser le hasard dans les petits départements (fonction `taux_cancer`).

Chaque département reçoit ensuite le tabagisme de sa région (`construire_table`). On prend le tabac de **2000**, la mesure la plus ancienne disponible.

In [ ]:
ANNEES_CANCER = [2022, 2023, 2024]

cancer_dep = taux_cancer(ameli, ANNEES_CANCER)
table = construire_table(cancer_dep, tabac_par_region(tabac, 2000), contours)

print(table.shape)
print(table.drop(columns="geometry").isna().sum())
table[["taux_brut", "taux_std"]].describe().round(0)

96 départements métropolitains, aucun manquant pour le cancer. Les deux départements corses n'ont pas de donnée de tabac, ils seront exclus des comparaisons avec le tabac.

La standardisation resserre fortement les écarts : l'écart-type passe de 36 à 20 pour 100 000. Une bonne partie des différences brutes venait donc simplement de l'âge.